In [2]:
import polars as pl
import numpy as np
from pathlib import Path
from catboost import CatBoostRanker, Pool
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("/kaggle/input/datasets/fedorfesenko/train-interactions")
WORK = Path("/kaggle/working")

In [3]:
df_train = pl.read_parquet(DATA_DIR / "train_dataset.parquet")
df_val = pl.read_parquet(DATA_DIR / "val_dataset.parquet")
test_dataset = pl.read_parquet(DATA_DIR / "test_dataset.parquet")
df_test_orig = pl.read_parquet(DATA_DIR / "df_test.parquet")
df_ranking_val = pl.read_parquet(DATA_DIR / "df_ranking_val.parquet")

candgen_train_v2 = pl.read_parquet(DATA_DIR / "candgen_features_train_v2.parquet")
candgen_val_v2 = pl.read_parquet(DATA_DIR / "candgen_features_val_v2.parquet")
candgen_test_v2 = pl.read_parquet(DATA_DIR / "candgen_features_test_v2.parquet")

print(f"df_train: {df_train.shape}")
print(f"df_val: {df_val.shape}")
print(f"test_dataset: {test_dataset.shape}")
print(f"df_test_orig: {df_test_orig.shape}")
print(f"candgen_train_v2: {candgen_train_v2.shape}")

df_train: (1810569, 74)
df_val: (323736, 74)
test_dataset: (292826, 72)
df_test_orig: (292826, 11)
candgen_train_v2: (15769472, 17)


In [8]:
key_cols = ["user_id", "product_id"]
candgen_train_v2 = candgen_train_v2.unique(subset=key_cols, keep="first")

candgen_train_v2 = candgen_train_v2.drop(["als_score", "v5_score", "v6_score"])

df_train_v5 = df_train.join(candgen_train_v2, on=key_cols, how="left")
print(f"после candgen: {df_train_v5.shape}")

df_ranking_train = pl.read_parquet(DATA_DIR / "df_ranking_train.parquet")
cat_train = (
    df_ranking_train
    .select(["user_id", "product_id", "request_id", "product_category"])
    .unique(subset=["user_id", "product_id", "request_id"], keep="first")
)
df_train_v5 = df_train_v5.join(cat_train, on=["user_id", "product_id", "request_id"], how="left")
print(f"после product_category: {df_train_v5.shape}")
print(f"product_category null: {df_train_v5['product_category'].null_count()}")

df_train_v5 = df_train_v5.with_columns([
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.hour().alias("hour_of_day"),
    pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.weekday().alias("day_of_week"),
])

print(f"\ndf_train_v5: {df_train_v5.shape}")
print(f"columns: {df_train_v5.columns}")

del df_ranking_train, cat_train
import gc; gc.collect()

после candgen: (1810569, 86)
после product_category: (1810569, 87)
product_category null: 0

df_train_v5: (1810569, 87)
columns: ['user_id', 'product_id', 'request_id', 'n_views_all', 'n_clicks_all', 'n_cart_updates_all', 'n_purchases_all', 'n_views_30d', 'n_clicks_30d', 'n_cart_updates_30d', 'n_purchases_30d', 'n_views_7d', 'n_clicks_7d', 'n_cart_updates_7d', 'n_purchases_7d', 'n_views_1d', 'n_clicks_1d', 'n_cart_updates_1d', 'n_purchases_1d', 'days_since_first_action', 'days_since_last_action', 'item_n_views_all', 'item_n_clicks_all', 'item_n_cart_updates_all', 'item_n_purchases_all', 'item_n_views_30d', 'item_n_clicks_30d', 'item_n_cart_updates_30d', 'item_n_purchases_30d', 'item_n_views_7d', 'item_n_clicks_7d', 'item_n_cart_updates_7d', 'item_n_purchases_7d', 'item_n_views_1d', 'item_n_clicks_1d', 'item_n_cart_updates_1d', 'item_n_purchases_1d', 'item_n_unique_users_all', 'item_n_unique_users_30d', 'item_n_unique_users_7d', 'item_n_unique_users_1d', 'item_days_since_first_view', 'i

0

In [9]:
candgen_val_v2 = candgen_val_v2.unique(subset=key_cols, keep="first")
candgen_val_v2 = candgen_val_v2.drop(["als_score", "v5_score", "v6_score"])

df_val_v5 = df_val.join(candgen_val_v2, on=key_cols, how="left")
print(f"после candgen: {df_val_v5.shape}")

if "product_category" not in df_val_v5.columns:
    cat_val = df_ranking_val.select(["user_id", "product_id", "request_id", "product_category"]).unique()
    df_val_v5 = df_val_v5.join(cat_val, on=["user_id", "product_id", "request_id"], how="left")
    print("Added product_category to val")

print(f"df_val_v5: {df_val_v5.shape}")
print(f"product_category null: {df_val_v5['product_category'].null_count()}")

после candgen: (323736, 86)
Added product_category to val
df_val_v5: (323736, 87)
product_category null: 0


In [10]:
key_cols_3 = ["user_id", "product_id", "request_id"]
candgen_test_v2 = candgen_test_v2.unique(subset=key_cols, keep="first")
candgen_test_v2 = candgen_test_v2.drop(["als_score", "v5_score", "v6_score"])

test_features_dedup = test_dataset.unique(subset=key_cols_3, keep="first")
cols_to_drop = [c for c in ["index", "timestamp", "source_type", "city_name"] if c in test_features_dedup.columns]
test_features_dedup = test_features_dedup.drop(cols_to_drop)

df_test_v5 = df_test_orig.join(test_features_dedup, on=key_cols_3, how="left", suffix="_features")
drop_right = [c for c in df_test_v5.columns if c.endswith("_right") or c.endswith("_features")]
if drop_right:
    df_test_v5 = df_test_v5.drop(drop_right)

df_test_v5 = df_test_v5.join(candgen_test_v2, on=key_cols, how="left")

if "hour_of_day" not in df_test_v5.columns:
    df_test_v5 = df_test_v5.with_columns([
        pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.hour().alias("hour_of_day"),
        pl.from_epoch(pl.col("timestamp"), time_unit="s").dt.weekday().alias("day_of_week"),
    ])

print(f"df_test_v5: {df_test_v5.shape}")
print(f"unique index: {df_test_v5['index'].n_unique()}")

df_test_v5: (292826, 88)
unique index: 292826


In [11]:
category_stats = (
    df_train_v5
    .group_by("product_category")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / df_train_v5.height).alias("freq"))
    .with_columns(pl.min_horizontal(-pl.col("freq").log(), pl.lit(5.0)).alias("category_idf"))
    .select(["product_category", "category_idf"])
)

df_train_v9 = df_train_v5.join(category_stats, on="product_category", how="left")
df_val_v9   = df_val_v5.join(category_stats, on="product_category", how="left")
df_test_v9  = df_test_v5.join(category_stats, on="product_category", how="left")

def add_tfidf(df):
    return df.with_columns([
        (pl.col("uc_n_cart_updates_all") * pl.col("category_idf")).alias("tfidf_cart"),
        (pl.col("uc_n_views_all") * pl.col("category_idf")).alias("tfidf_views"),
        (pl.col("uc_n_purchases_all") * pl.col("category_idf")).alias("tfidf_purchases"),
        (pl.col("uc_n_clicks_all") * pl.col("category_idf")).alias("tfidf_clicks"),
    ])

df_train_v9 = add_tfidf(df_train_v9)
df_val_v9   = add_tfidf(df_val_v9)
df_test_v9  = add_tfidf(df_test_v9)

print(f"df_train_v9: {df_train_v9.shape}")
print(f"df_val_v9:   {df_val_v9.shape}")
print(f"df_test_v9:  {df_test_v9.shape}")

df_train_v9: (1810569, 92)
df_val_v9:   (323736, 92)
df_test_v9:  (292826, 93)


In [13]:
BASE_NUM = [
    "n_views_all", "n_clicks_all", "n_cart_updates_all", "n_purchases_all",
    "n_views_30d", "n_clicks_30d", "n_cart_updates_30d", "n_purchases_30d",
    "n_views_7d", "n_clicks_7d", "n_cart_updates_7d", "n_purchases_7d",
    "n_views_1d", "n_clicks_1d", "n_cart_updates_1d", "n_purchases_1d",
    "days_since_first_action", "days_since_last_action",
    "item_n_views_all", "item_n_clicks_all", "item_n_cart_updates_all", "item_n_purchases_all",
    "item_n_views_30d", "item_n_clicks_30d", "item_n_cart_updates_30d", "item_n_purchases_30d",
    "item_n_views_7d", "item_n_clicks_7d", "item_n_cart_updates_7d", "item_n_purchases_7d",
    "item_n_views_1d", "item_n_clicks_1d", "item_n_cart_updates_1d", "item_n_purchases_1d",
    "item_n_unique_users_all", "item_n_unique_users_30d", "item_n_unique_users_7d", "item_n_unique_users_1d",
    "item_days_since_first_view", "item_days_since_last_view",
    "ui_n_views_all", "ui_n_clicks_all", "ui_n_cart_updates_all", "ui_n_purchases_all",
    "ui_days_since_last_action", "ui_days_since_last_cart_update", "ui_days_since_last_purchase",
    "uc_n_views_all", "uc_n_clicks_all", "uc_n_cart_updates_all", "uc_n_purchases_all",
    "uc_n_views_30d", "uc_n_clicks_30d", "uc_n_cart_updates_30d", "uc_n_purchases_30d",
    "uc_n_views_7d", "uc_n_clicks_7d", "uc_n_cart_updates_7d", "uc_n_purchases_7d",
    "uc_days_since_last_purchase",
    "als_score", "v5_score", "v6_score",
    "hour_of_day", "day_of_week",
    "als_user_norm", "als_item_norm", "als_cosine", "als_norms_prod",
    "v5_user_norm",  "v5_item_norm",  "v5_cosine",  "v5_norms_prod",
    "v6_user_norm",  "v6_item_norm",  "v6_cosine",  "v6_norms_prod",
]
CAT_FEATURES = ["source_type", "city_name", "product_category"]
FEATURES = BASE_NUM + CAT_FEATURES

FEATURES_V9 = FEATURES + ["category_idf", "tfidf_cart", "tfidf_views", "tfidf_purchases", "tfidf_clicks"]
print(f"FEATURES_V9: {len(FEATURES_V9)}")

for name, df in [("train", df_train_v9), ("val", df_val_v9), ("test", df_test_v9)]:
    missing = [f for f in FEATURES_V9 if f not in df.columns]
    print(f"missing in {name}: {missing}")

def prepare_pool(df, features, cat_features, target_col="target", group_col="request_id"):
    df = df.sort(group_col)
    X = df.select(features).to_pandas()
    for c in features:
        if c not in cat_features:
            X[c] = X[c].astype("float32")
    for c in cat_features:
        X[c] = X[c].fillna("unknown").astype(str)
    if target_col in df.columns:
        y = df[target_col].to_numpy().astype("float32")
    else:
        y = np.zeros(len(df), dtype="float32")
    group_id = df[group_col].to_numpy()
    return Pool(data=X, label=y, group_id=group_id, cat_features=cat_features), df

train_pool_v9, _ = prepare_pool(df_train_v9, FEATURES_V9, CAT_FEATURES)
val_pool_v9, _ = prepare_pool(df_val_v9, FEATURES_V9, CAT_FEATURES)
test_pool_v9, df_test_sorted_v9 = prepare_pool(df_test_v9, FEATURES_V9, CAT_FEATURES)

print(f"train: {train_pool_v9.num_row()}, val: {val_pool_v9.num_row()}, test: {test_pool_v9.num_row()}")

FEATURES_V9: 85
missing in train: []
missing in val: []
missing in test: []
train: 1810569, val: 323736, test: 292826


In [14]:
model_v9 = CatBoostRanker(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    task_type="CPU",
    thread_count=-1,
    verbose=100,
    random_seed=42,
    allow_writing_files=False,
)
model_v9.fit(train_pool_v9, eval_set=val_pool_v9, early_stopping_rounds=50, use_best_model=True, verbose=100)

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.8462861	best: 0.8462861 (0)	total: 2.44s	remaining: 40m 43s
100:	test: 0.8814930	best: 0.8817914 (78)	total: 4m 41s	remaining: 41m 44s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8829083235
bestIteration = 119

Shrink model to first 120 iterations.


CatBoostRanker(allow_writing_files=False, depth=8, eval_metric='NDCG:top=10', iterations=1000, learning_rate=0.05, loss_function='YetiRank', random_seed=42, task_type='CPU', verbose=100)

In [15]:
test_preds_v9 = model_v9.predict(test_pool_v9)
df_sub_v9 = pl.DataFrame({
    "index": df_test_sorted_v9["index"].to_numpy(),
    "request_id": df_test_sorted_v9["request_id"].to_numpy(),
    "score": test_preds_v9.astype("float64"),
})
df_sub_v9_sorted = df_sub_v9.sort(["request_id", "score"], descending=[False, True])
df_sub_v9_final = df_sub_v9_sorted.select(["index", "request_id"])
df_sub_v9_final.write_csv("submission_v9.csv")

print(f"submission_v9.csv: {df_sub_v9_final.shape}")
print(f"unique index: {df_sub_v9_final['index'].n_unique()}")

submission_v9.csv: (292826, 2)
unique index: 292826


In [17]:
def add_new_features(df):
    user_conv = pl.col("n_cart_updates_all") / (pl.col("n_views_all") + 1e-6)
    user_purchase_conv = pl.col("n_purchases_all") / (pl.col("n_views_all") + 1e-6)
    
    item_conv = pl.col("item_n_cart_updates_all") / (pl.col("item_n_views_all") + 1e-6)
    item_purchase_conv = pl.col("item_n_purchases_all") / (pl.col("item_n_views_all") + 1e-6)
    
    ui_to_user_cart = pl.col("ui_n_cart_updates_all") / (pl.col("n_cart_updates_all") + 1e-6)
    ui_to_user_views = pl.col("ui_n_views_all") / (pl.col("n_views_all") + 1e-6)
    ui_to_item_cart = pl.col("ui_n_cart_updates_all") / (pl.col("item_n_cart_updates_all") + 1e-6)
    
    df = df.with_columns([
        user_conv.alias("user_cart_conv"),
        user_purchase_conv.alias("user_purchase_conv"),
        item_conv.alias("item_cart_conv"),
        item_purchase_conv.alias("item_purchase_conv"),
        ui_to_user_cart.alias("ui_to_user_cart"),
        ui_to_user_views.alias("ui_to_user_views"),
        ui_to_item_cart.alias("ui_to_item_cart"),
    ])
    
    df = df.with_columns([
        pl.col("als_score").rank("ordinal").over("request_id").alias("als_rank"),
        pl.col("v6_score").rank("ordinal").over("request_id").alias("v6_rank"),
        pl.col("als_cosine").rank("ordinal").over("request_id").alias("als_cosine_rank"),
    ])
    
    df = df.with_columns([
        pl.col("als_score").mean().over("request_id").alias("als_mean_group"),
        pl.col("v6_score").mean().over("request_id").alias("v6_mean_group"),
    ]).with_columns([
        (pl.col("als_score") - pl.col("als_mean_group")).alias("als_dev_from_mean"),
        (pl.col("v6_score") - pl.col("v6_mean_group")).alias("v6_dev_from_mean"),
    ]).drop(["als_mean_group", "v6_mean_group"])
    
    return df

df_train_v11 = add_new_features(df_train_v9)
df_val_v11 = add_new_features(df_val_v9)
df_test_v11 = add_new_features(df_test_v9)

print(f"df_train_v11: {df_train_v11.shape}")
print(f"df_val_v11:   {df_val_v11.shape}")
print(f"df_test_v11:  {df_test_v11.shape}")

df_train_v11: (1810569, 104)
df_val_v11:   (323736, 104)
df_test_v11:  (292826, 105)


In [18]:
NEW_FEATS = [
    "user_cart_conv", "user_purchase_conv",
    "item_cart_conv", "item_purchase_conv",
    "ui_to_user_cart", "ui_to_user_views", "ui_to_item_cart",
    "als_rank", "v6_rank", "als_cosine_rank",
    "als_dev_from_mean", "v6_dev_from_mean",
]
FEATURES_V11 = FEATURES_V9 + NEW_FEATS
print(f"FEATURES_V11: {len(FEATURES_V11)}")

for name, df in [("train", df_train_v11), ("val", df_val_v11), ("test", df_test_v11)]:
    missing = [f for f in FEATURES_V11 if f not in df.columns]
    print(f"missing in {name}: {missing}")

train_pool_v11, _ = prepare_pool(df_train_v11, FEATURES_V11, CAT_FEATURES)
val_pool_v11, _ = prepare_pool(df_val_v11, FEATURES_V11, CAT_FEATURES)
test_pool_v11, df_test_sorted_v11 = prepare_pool(df_test_v11, FEATURES_V11, CAT_FEATURES)

print(f"train: {train_pool_v11.num_row()}, val: {val_pool_v11.num_row()}, test: {test_pool_v11.num_row()}")

FEATURES_V11: 97
missing in train: []
missing in val: []
missing in test: []
train: 1810569, val: 323736, test: 292826


In [19]:
model_v11 = CatBoostRanker(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    task_type="CPU",
    thread_count=-1,
    verbose=100,
    random_seed=42,
    allow_writing_files=False,
)
model_v11.fit(train_pool_v11, eval_set=val_pool_v11, early_stopping_rounds=50, use_best_model=True, verbose=100)

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.8531286	best: 0.8531286 (0)	total: 2.43s	remaining: 40m 24s
100:	test: 0.8812653	best: 0.8812653 (100)	total: 4m 57s	remaining: 44m 8s
200:	test: 0.8818446	best: 0.8818585 (190)	total: 9m 38s	remaining: 38m 20s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8818584615
bestIteration = 190

Shrink model to first 191 iterations.


CatBoostRanker(allow_writing_files=False, depth=8, eval_metric='NDCG:top=10', iterations=1000, learning_rate=0.05, loss_function='YetiRank', random_seed=42, task_type='CPU', verbose=100)

In [20]:
test_preds_v11 = model_v11.predict(test_pool_v11)
df_sub_v11 = pl.DataFrame({
    "index": df_test_sorted_v11["index"].to_numpy(),
    "request_id": df_test_sorted_v11["request_id"].to_numpy(),
    "score": test_preds_v11.astype("float64"),
})
df_sub_v11_sorted = df_sub_v11.sort(["request_id", "score"], descending=[False, True])
df_sub_v11_final = df_sub_v11_sorted.select(["index", "request_id"])
df_sub_v11_final.write_csv("submission_v11.csv")
print(f"submission_v11.csv: {df_sub_v11_final.shape}")
print(f"unique index: {df_sub_v11_final['index'].n_unique()}")

submission_v11.csv: (292826, 2)
unique index: 292826


In [22]:
imp_v11 = model_v11.get_feature_importance(data=train_pool_v11, type="FeatureImportance")
imp_df_v11 = pl.DataFrame({
    "feature": model_v11.feature_names_,
    "importance": imp_v11,
}).sort("importance", descending=True)

print("=== Top-30 feature importance (v11) ===")
print(imp_df_v11.head(30))

print("\n=== Новые фичи из v11 (ранги, отношения, конверсии) ===")
new_feats_v11 = [
    "user_cart_conv", "user_purchase_conv",
    "item_cart_conv", "item_purchase_conv",
    "ui_to_user_cart", "ui_to_user_views", "ui_to_item_cart",
    "als_rank", "v6_rank", "als_cosine_rank",
    "als_dev_from_mean", "v6_dev_from_mean",
]
new_imp = imp_df_v11.filter(pl.col("feature").is_in(new_feats_v11))
print(new_imp)

print(f"\nСуммарная важность новых фич: {new_imp['importance'].sum():.4f}")

print("\n=== Бесполезные фичи (importance < 0.0001) ===")
useless = imp_df_v11.filter(pl.col("importance") < 0.0001)
print(useless)

=== Top-30 feature importance (v11) ===
shape: (30, 2)
┌────────────────────────────┬────────────┐
│ feature                    ┆ importance │
│ ---                        ┆ ---        │
│ str                        ┆ f64        │
╞════════════════════════════╪════════════╡
│ v6_score                   ┆ 0.03773    │
│ source_type                ┆ 0.023772   │
│ als_score                  ┆ 0.023711   │
│ als_cosine                 ┆ 0.012178   │
│ product_category           ┆ 0.009633   │
│ …                          ┆ …          │
│ item_days_since_first_view ┆ 0.000307   │
│ uc_n_cart_updates_all      ┆ 0.000252   │
│ tfidf_cart                 ┆ 0.000201   │
│ uc_n_cart_updates_7d       ┆ 0.000194   │
│ v5_cosine                  ┆ 0.000135   │
└────────────────────────────┴────────────┘

=== Новые фичи из v11 (ранги, отношения, конверсии) ===
shape: (12, 2)
┌────────────────────┬────────────┐
│ feature            ┆ importance │
│ ---                ┆ ---        │
│ str            

In [25]:
model_v12 = CatBoostRanker(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    task_type="CPU",
    thread_count=-1,
    verbose=100,
    random_seed=42,
    allow_writing_files=False,
)
model_v12.fit(train_pool_v12, eval_set=val_pool_v12, early_stopping_rounds=50, use_best_model=True, verbose=100)

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.8507615	best: 0.8507615 (0)	total: 2.73s	remaining: 45m 28s
100:	test: 0.8800929	best: 0.8805344 (80)	total: 4m 15s	remaining: 37m 52s
200:	test: 0.8810104	best: 0.8811123 (198)	total: 8m 4s	remaining: 32m 6s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8811123189
bestIteration = 198

Shrink model to first 199 iterations.


CatBoostRanker(allow_writing_files=False, depth=8, eval_metric='NDCG:top=10', iterations=1000, learning_rate=0.05, loss_function='YetiRank', random_seed=42, task_type='CPU', verbose=100)

In [26]:
test_preds_v12 = model_v12.predict(test_pool_v12)
df_sub_v12 = pl.DataFrame({
    "index": df_test_sorted_v12["index"].to_numpy(),
    "request_id": df_test_sorted_v12["request_id"].to_numpy(),
    "score": test_preds_v12.astype("float64"),
})
df_sub_v12_sorted = df_sub_v12.sort(["request_id", "score"], descending=[False, True])
df_sub_v12_final = df_sub_v12_sorted.select(["index", "request_id"])
df_sub_v12_final.write_csv("submission_v12.csv")
print(f"submission_v12.csv: {df_sub_v12_final.shape}")
print(f"unique index: {df_sub_v12_final['index'].n_unique()}")

submission_v12.csv: (292826, 2)
unique index: 292826
